# SRAにおける「仮想ニューロン」創発検証実験 v2

このノートブックでは、**文字分布が根本的に異なる5つのドメイン × 5タスク（計25タスク）** をSRAモデルに同時学習させます。

- **英語テキスト (NL)** — a-z, スペース, 句読点
- **Pythonコード (CODE)** — def, :, =, (, #, インデント
- **数式テキスト (MATH)** — 数字, +, -, *, /, =
- **DNA配列 (DNA)** — A, T, G, C のみ
- **CSVデータ (CSV)** — 数字, カンマ

各ドメインの入力は文字の種類・分布が全く異なるため、**タスクIDなどのヒントなしに**ルーターが自律的にタスクを識別し、ドメインごとに異なるシナプス群（＝仮想ニューロン）を形成するかを検証します。

In [ ]:
import sys
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random
import numpy as np
import collections

if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn numpy nbformat

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_optimizer, load_balance_loss, specialization_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. タスク設計とデータジェネレータ

5ドメイン × 5タスクを定義します。各ドメインは入力の文字分布が根本的に異なります。
モデルはキャラクターレベル（ASCII）で動作し、入力テキストを文字→ASCII整数にエンコードして処理します。

In [ ]:
import random, string

# ===== キャラクターレベル エンコーダー =====
VOCAB_SIZE = 128  # ASCII
PAD = 0
BOS = 1
EOS = 2

def encode(text):
    """ASCII文字列 → intリスト (BOS付き)"""
    return [BOS] + [ord(c) for c in text] + [EOS]

def decode(ids):
    """intリスト → 文字列 (制御トークン除外)"""
    return ''.join(chr(i) for i in ids if i >= 32)

def pad_to(seq, length):
    return seq[:length] + [PAD] * max(0, length - len(seq))

# ===== ドメイン1: 英語テキスト (NL) =====
WORDS = ["hello", "world", "python", "learn", "great", "smart", "open", "code",
         "data", "fast", "model", "brain", "build", "train", "green", "black"]

def nl_upper(w=None):
    w = w or random.choice(WORDS)
    return w, w.upper()

def nl_vowel_count(w=None):
    w = w or random.choice(WORDS)
    c = sum(1 for ch in w if ch in 'aeiou')
    return w, str(c)

def nl_reverse(w=None):
    w = w or random.choice(WORDS)
    return w, w[::-1]

def nl_length(w=None):
    w = w or random.choice(WORDS)
    return w, str(len(w))

def nl_first_char(w=None):
    w = w or random.choice(WORDS)
    return w, w[0]

# ===== ドメイン2: Pythonコード (CODE) =====
VARS = ["x", "y", "z", "n", "val", "res", "cnt", "idx"]
OPS  = ["+", "-", "*"]

def code_indent(v=None, op=None, n=None):
    v = v or random.choice(VARS); op = op or random.choice(OPS); n = n or random.randint(1,9)
    line = f"return {v} {op} {n}"
    return line, "    " + line

def code_is_comment():
    if random.random() < 0.5:
        line = f"# {random.choice(['add values','init','loop','check'])}"
        return line, "yes"
    else:
        v = random.choice(VARS); n = random.randint(1,9)
        return f"{v} = {n}", "no"

def code_operator(v=None, op=None, n=None):
    v = v or random.choice(VARS); op = op or random.choice(OPS); n = n or random.randint(1,9)
    return f"{v} = a {op} {n}", op

def code_varname(v=None, n=None):
    v = v or random.choice(VARS); n = n or random.randint(1,99)
    return f"{v} = {n}", v

def code_has_return():
    if random.random() < 0.5:
        v = random.choice(VARS)
        return f"return {v}", "yes"
    else:
        v = random.choice(VARS); n = random.randint(1,9)
        return f"{v} = {n}", "no"

# ===== ドメイン3: 数式 (MATH) =====
def math_add():
    a, b = random.randint(1,19), random.randint(1,19)
    return f"{a}+{b}=", str(a+b)

def math_sub():
    a, b = random.randint(1,19), random.randint(1,19)
    lo, hi = min(a,b), max(a,b)
    return f"{hi}-{lo}=", str(hi-lo)

def math_mul():
    a, b = random.randint(1,9), random.randint(1,9)
    return f"{a}*{b}=", str(a*b)

def math_gt():
    a, b = random.randint(1,19), random.randint(1,19)
    return f"{a}>{b}?", "yes" if a > b else "no"

def math_lt():
    a, b = random.randint(1,19), random.randint(1,19)
    return f"{a}<{b}?", "yes" if a < b else "no"

# ===== ドメイン4: DNA配列 (DNA) =====
BASES = ['A', 'T', 'G', 'C']
COMP  = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

def rand_dna(n=5):
    return ''.join(random.choices(BASES, k=n))

def dna_complement():
    s = rand_dna()
    return s, ''.join(COMP[c] for c in s)

def dna_reverse():
    s = rand_dna()
    return s, s[::-1]

def dna_repeat():
    s = rand_dna(3)
    return s, s * 2

def dna_count_g():
    s = rand_dna()
    return s, str(s.count('G'))

def dna_has_a():
    s = rand_dna()
    return s, "yes" if 'A' in s else "no"

# ===== ドメイン5: CSVデータ (CSV) =====
def rand_csv(n=4):
    nums = [random.randint(1, 20) for _ in range(n)]
    return ','.join(str(x) for x in nums), nums

def csv_count():
    s, nums = rand_csv()
    return s, str(len(nums))

def csv_max():
    s, nums = rand_csv()
    return s, str(max(nums))

def csv_min():
    s, nums = rand_csv()
    return s, str(min(nums))

def csv_first():
    s, nums = rand_csv()
    return s, str(nums[0])

def csv_last():
    s, nums = rand_csv()
    return s, str(nums[-1])

# ===== タスク辞書 =====
def make_sample(task_fn):
    result = task_fn()
    return result  # (input_str, output_str)

ALL_TASKS = {
    # NL
    "nl_upper":       lambda: nl_upper(),
    "nl_vowel_count": lambda: nl_vowel_count(),
    "nl_reverse":     lambda: nl_reverse(),
    "nl_length":      lambda: nl_length(),
    "nl_first_char":  lambda: nl_first_char(),
    # CODE
    "code_indent":    lambda: code_indent(),
    "code_is_comment":lambda: code_is_comment(),
    "code_operator":  lambda: code_operator(),
    "code_varname":   lambda: code_varname(),
    "code_has_return":lambda: code_has_return(),
    # MATH
    "math_add":       lambda: math_add(),
    "math_sub":       lambda: math_sub(),
    "math_mul":       lambda: math_mul(),
    "math_gt":        lambda: math_gt(),
    "math_lt":        lambda: math_lt(),
    # DNA
    "dna_complement": lambda: dna_complement(),
    "dna_reverse":    lambda: dna_reverse(),
    "dna_repeat":     lambda: dna_repeat(),
    "dna_count_g":    lambda: dna_count_g(),
    "dna_has_a":      lambda: dna_has_a(),
    # CSV
    "csv_count":      lambda: csv_count(),
    "csv_max":        lambda: csv_max(),
    "csv_min":        lambda: csv_min(),
    "csv_first":      lambda: csv_first(),
    "csv_last":       lambda: csv_last(),
}

DOMAIN_COLORS = {
    "nl":   "#4C72B0",
    "code": "#DD8452",
    "math": "#55A868",
    "dna":  "#C44E52",
    "csv":  "#8172B3",
}

def get_domain(task_name):
    return task_name.split('_')[0]

print(f"Total tasks: {len(ALL_TASKS)}")
for domain in ['nl','code','math','dna','csv']:
    tasks = [t for t in ALL_TASKS if get_domain(t)==domain]
    print(f"  {domain}: {tasks}")

# 動作確認
for name, fn in list(ALL_TASKS.items())[:5]:
    inp, out = fn()
    print(f"[{name}] '{inp}' → '{out}'")


## 2. モデル設定と学習

キャラクターレベル（ASCII 128）でSRAモデルを初期化し、25タスクを同時学習させます。
入力テキストの文字分布が根本的に異なるため、ルーターはタスクIDなしで自律的に各タスクを識別して別々のシナプスに振り分けるはずです。

In [ ]:
# モデル設定
MAX_SEQ_LEN = 24   # 最大入力文字数（入力+出力）
VOCAB_SIZE   = 128 # ASCII

DIM          = 64
LAYERS       = 2
NUM_SYNAPSES = 32
K            = 2
SYN_HIDDEN   = 128

model = MoESRAModel(
    vocab_size   = VOCAB_SIZE,
    dim          = DIM,
    layers       = LAYERS,
    num_synapses = NUM_SYNAPSES,
    k            = K,
    syn_hidden   = SYN_HIDDEN,
).to(device)
optimizer = make_optimizer(model, lr=1e-3)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Vocab size: {VOCAB_SIZE} (ASCII character-level)")
print(f"Synapses: {NUM_SYNAPSES}, k={K}")

def make_multitask_batch(tasks, batch_size, max_len=MAX_SEQ_LEN):
    """各タスクから均等にサンプルし、テキストをエンコードしてバッチ化する"""
    xs, ys = [], []
    task_names = list(tasks.keys())
    for _ in range(batch_size):
        task_name = random.choice(task_names)
        inp_str, out_str = tasks[task_name]()
        x = pad_to(encode(inp_str), max_len)
        y = pad_to(encode(out_str), max_len)
        xs.append(x)
        ys.append(y)
    x = torch.tensor(xs, dtype=torch.long, device=device)
    y = torch.tensor(ys, dtype=torch.long, device=device)
    return x, y


In [ ]:
# 学習ループ
print("Multitask Training on 25 Tasks started...")
model.train()

epochs     = 2000
batch_size = 128

for epoch in range(epochs):
    x, y = make_multitask_batch(ALL_TASKS, batch_size)
    
    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model(x, y_in)
    
    loss    = F.cross_entropy(outputs.reshape(-1, VOCAB_SIZE), y.reshape(-1), ignore_index=PAD)
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | LB Loss: {lb_loss.item():.4f}")

print("Training finished!")


## 3. 仮想ニューロンの可視化と解析

学習後、各タスクごとにルーターがどのシナプスを選択したかを抽出します。これをドメインごとにまとめてヒートマップ化することで、シナプスの使われ方に共通性があるか（仮想ニューロンの創発）を確認します。

In [ ]:
def get_task_routing_vector(task_name, task_fn, samples=10):
    """テキストタスクのルーティングベクトルを取得する"""
    model.eval()
    vectors = []
    with torch.no_grad():
        for _ in range(samples):
            inp_str, out_str = task_fn()
            x = torch.tensor([pad_to(encode(inp_str), MAX_SEQ_LEN)], dtype=torch.long, device=device)
            y_dummy = torch.tensor([pad_to(encode(out_str), MAX_SEQ_LEN)], dtype=torch.long, device=device)
            y_in = torch.cat([torch.full((1, 1), BOS, dtype=torch.long, device=device), y_dummy[:, :-1]], dim=1)
            _, routing_weights, _ = model(x, y_in)
            avg_rw = routing_weights[-1].squeeze(0).mean(dim=0).cpu().numpy()
            vectors.append(avg_rw)
    return np.array(vectors).mean(axis=0)

# 全タスクのルーティングベクトルを収集
task_vectors = {}
for task_name, task_fn in ALL_TASKS.items():
    task_vectors[task_name] = get_task_routing_vector(task_name, task_fn)

# ドメイン順に並べたタスクリスト
ordered_tasks = (
    [t for t in ALL_TASKS if get_domain(t)=='nl']   +
    [t for t in ALL_TASKS if get_domain(t)=='code'] +
    [t for t in ALL_TASKS if get_domain(t)=='math'] +
    [t for t in ALL_TASKS if get_domain(t)=='dna']  +
    [t for t in ALL_TASKS if get_domain(t)=='csv']
)

heatmap_data = np.array([task_vectors[t] for t in ordered_tasks])
row_colors   = [DOMAIN_COLORS[get_domain(t)] for t in ordered_tasks]

fig, (ax_color, ax) = plt.subplots(1, 2, figsize=(16, 10),
                                    gridspec_kw={'width_ratios': [0.04, 1]})

# ドメインカラーバー
color_matrix = np.zeros((len(ordered_tasks), 1, 3))
import matplotlib.colors as mcolors
for i, t in enumerate(ordered_tasks):
    rgb = mcolors.to_rgb(DOMAIN_COLORS[get_domain(t)])
    color_matrix[i, 0] = rgb

ax_color.imshow(color_matrix, aspect='auto')
ax_color.set_xticks([])
ax_color.set_yticks(range(len(ordered_tasks)))
ax_color.set_yticklabels(ordered_tasks, fontsize=8)

# ヒートマップ
sns.heatmap(heatmap_data, ax=ax, cmap='Blues',
            xticklabels=range(NUM_SYNAPSES),
            yticklabels=ordered_tasks, linewidths=0.3)
ax.set_xlabel("Synapse Index", fontsize=12)
ax.set_title("Synapse Routing Heatmap by Domain\n(5 Domains × 5 Tasks, Character-Level)", fontsize=13)
ax.set_yticklabels(ordered_tasks, fontsize=8)

# 凡例
import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=c, label=d.upper())
                  for d, c in DOMAIN_COLORS.items()]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig("virtual_neuron_heatmap_v2.png", dpi=150, bbox_inches='tight')
plt.show()
print("Heatmap saved as virtual_neuron_heatmap_v2.png")


In [ ]:
# 仮想ニューロン（同じシナプスの組み合わせを使うタスク群）の抽出と表示

virtual_neurons = collections.defaultdict(list)

for task, vector in task_vectors.items():
    # ルーティング確率が一定以上（ここでは0.1）のシナプスを「使用しているシナプス」とみなす
    active_synapses = tuple(int(x) for x in sorted(np.where(vector >= 0.1)[0]))
    virtual_neurons[active_synapses].append(task)

# 含まれるタスクが多い順にソート
sorted_neurons = sorted(virtual_neurons.items(), key=lambda x: len(x[1]), reverse=True)

print("=== 抽出された仮想ニューロン（Cell Assemblies） ===")
print("※「同じシナプスの組み合わせ」を使用しているタスクを1つの仮想ニューロンとしてグループ化\n")

for i, (synapses, tasks) in enumerate(sorted_neurons, 1):
    print(f"[Virtual Neuron {i}]")
    print(f"  Active Synapses (Combination) : {synapses}")
    print(f"  Belonging Tasks ({len(tasks)} tasks):")
    for t in tasks:
        print(f"    - {t}")
    print()


# シナプスの出現回数をカウント（複数の仮想ニューロンにまたがるか）
synapse_counts = collections.defaultdict(int)
for synapses, _ in sorted_neurons:
    for syn in synapses:
        synapse_counts[syn] += 1

shared_synapses = sorted([syn for syn, count in synapse_counts.items() if count > 1])
unique_synapses = sorted([syn for syn, count in synapse_counts.items() if count == 1])

print("=== シナプスの共有状況（知識のエンタングルメント分析） ===")
print(f"複数の仮想ニューロンで共有されているシナプス（基礎・共通モジュール）:")
print(f"  {shared_synapses}")
print(f"\n単一の仮想ニューロンでのみ使用されるシナプス（高度な特化モジュール）:")
print(f"  {unique_synapses}")


このように、**「完全に同じシナプスの組み合わせ」を使っているタスク群が、ネットワーク内で1つの独立した『仮想ニューロン』を形成している**ことが確認できます。類似タスクは同じ仮想ニューロンに属し、全く異なるタスクは別の仮想ニューロンを形成します。

### 考察ポイント
* **共通シナプスと個別シナプス**：多くのドメインで広く発火しているシナプス（共通コア）と、同じドメイン内のタスクで共通して強く発火しているシナプス（専門・個別）があるかを確認します。
* **仮想ニューロン（Cell Assembly）の形成**：それらの「共通シナプス」と「個別シナプス」の**発火の組み合わせパターン（まとまり）**が、そのドメイン・タスクを実行するための「仮想ニューロン」として機能していると言えます。
* **独立した発火**：置換暗号など全く異なるルールのタスクでは、他のタスクとは異なる独立したシナプス群の組み合わせが使われているかを観察してみてください。